# Similarity scores and representative covers

Reconstruct selected pairwise scores, read part of a complete score table, and query representative assignments. {ref}`Similarity scores <similarity-score-reference>` defines the metrics and files.

In [ ]:
import os

os.environ.setdefault("PLINDER_RELEASE", "2026-07")
os.environ.setdefault("PLINDER_RELEASE_NUMBER", "1")

In [ ]:
import pandas as pd

from plinder.core import PlinderRelease, query_table
from plinder.core.scores import (
    reconstruct_interface_similarity_scores,
    reconstruct_similarity_scores,
)

release = PlinderRelease()

In [ ]:
ligand_system_id = "4agi__1__1.C__1.R"
ligand_id = "4agi__1__1.R"
interface_id = "7cm8__1__1.A--2.A"

## Reconstruct ligand similarities

`reconstruct_similarity_scores()` reads the files needed for the requested query and target IDs. This example uses a one-pair self-comparison to keep execution small.

In [ ]:
ligand_scores = reconstruct_similarity_scores(
    query_system_ids=[ligand_system_id],
    target_system_ids=[ligand_system_id],
    query_ligand_ids=[ligand_id],
    target_ligand_ids=[ligand_id],
)
ligand_scores

For larger neighborhoods, read selected columns and rows from the complete ligand score table.

In [ ]:
def read_ligand_neighborhood(system_id, ligand_id):
    return pd.read_parquet(
        release.fetch("ligand_similarity_scores"),
        columns=[
            "query_system",
            "query_ligand_id",
            "target_system",
            "target_ligand_id",
            "pocket_qcov",
            "pocket_fident_qcov",
            "pli_qcov",
            "sucos_shape",
        ],
        filters=[
            ("query_system", "==", system_id),
            ("query_ligand_id", "==", ligand_id),
        ],
    )

## Reconstruct protein-interface similarities

The interface API likewise accepts selected query and target IDs.

In [ ]:
interface_scores = reconstruct_interface_similarity_scores(
    query_interface_ids=[interface_id],
    target_interface_ids=[interface_id],
)
interface_scores

## Representative covers

Request cover columns through `query_table()` just like annotation columns. {ref}`Representative covers <representative-cover-reference>` describes their construction and long-form files.

In [ ]:
pocket_cover = "pocket_qcov__70__ligand__directed_set_cover"
ligand_cover = query_table(
    columns=[
        "system_id",
        "ligand_id",
        pocket_cover,
        f"{pocket_cover}__is_centroid",
    ],
    filters=[("ligand_id", "==", ligand_id)],
)
ligand_cover.T

In [ ]:
interface_cover = query_table(
    "interface_annotations",
    columns=[
        "system_id",
        "interface_qcov__70__directed_set_cover",
    ],
    filters=[("system_id", "==", interface_id)],
)
interface_cover.T

## Matched molecular pairs

Use `ligand_mmp_pairs` to look up transformations involving the selected ligand chemistry.

In [ ]:
smiles_id = query_table(
    columns=["ligand_smiles_id"],
    filters=[("ligand_id", "==", ligand_id)],
).iloc[0, 0]
mmp_neighbors = query_table(
    "ligand_mmp_pairs",
    filters=[[
        ("ligand_smiles_id_1", "==", smiles_id),
        ("ligand_smiles_id_2", "==", smiles_id),
    ]],
)
mmp_neighbors.head()